# Gruppe 4 Abgabe

## Import der wichtigsten Pakete

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import (
    proportion_confint,
    proportions_ztest,
    confint_proportions_2indep,
)

## Einlesen der Datensätze

In [39]:
df_categories = pd.read_csv("data/categories.csv")
df_customers = pd.read_csv("data/customers.csv")
df_geolocation = pd.read_csv("data/geolocation.csv")
df_items = pd.read_csv("data/items.csv")
df_orders= pd.read_csv("data/orders.csv")
df_payments = pd.read_csv("data/payments.csv")
df_pilot_support = pd.read_csv("data/pilot_support.csv")
df_products = pd.read_csv("data/products.csv")
df_reviews = pd.read_csv("data/reviews.csv")
df_sellers = pd.read_csv("data/sellers.csv")
df_service_experiment = pd.read_csv("data/service_experiment.csv")
df_service_observation = pd.read_csv("data/service_observation.csv")

# 2.1 Prüfung der Datensätze


## Funktion um Tabellen einheitlich zu Prüfen

In [ ]:
import pandas as pd

def pruefe_tabelle(df, name, schluessel, kategorien=None, fremdschluessel=None):

    # 1. Struktur und Datentypen
    print("Shape:", df.shape)
    df.info()

    # 2. Fehlende Werte (absolut und in %)
    na = pd.DataFrame({'fehlend': df.isna().sum(),
                       'prozent': (df.isna().mean() * 100).round(2)})
    print("\nFehlende Werte:\n", na[na['fehlend'] > 0] if na['fehlend'].any() else "keine")

    # 3. Duplikate
    print("\nDoppelte Zeilen:", df.duplicated().sum())
    print(f"Doppelte Schlüssel {schluessel}:", df.duplicated(subset=schluessel).sum())

    # 4. Wertebereiche der Zahlenspalten
    zahlen = df.select_dtypes(include='number')
    if not zahlen.empty:
        print("\nWertebereiche:\n", zahlen.describe().T)

    # 5. Kategorien
    for spalte in (kategorien or []):
        print(f"\nKategorien in {spalte}:\n", df[spalte].value_counts(dropna=False))

    # 6. Bezüge zu anderen Tabellen
    for spalte, (ziel_df, ziel_spalte) in (fremdschluessel or {}).items():
        werte = df[spalte].dropna()
        fehlend = (~werte.isin(ziel_df[ziel_spalte])).sum()
        print(f"\n{spalte} nicht in Zieltabelle ({ziel_spalte}): {fehlend} von {len(werte)}")

#Umwandlung in der Zeitspalten in DateTime-Format
def umwandlung_Zeitspalte(df, spalten, name):

    df = df.copy()
    na_vorher = df[spalten].isna().sum()

    for spalte in spalten:
        df[spalte] = pd.to_datetime(df[spalte], errors='coerce')

    neu_fehlend = df[spalten].isna().sum() - na_vorher
    print(f"{name}: neue fehlende Werte durch Umwandlung:\n{neu_fehlend}")

    return df

## Orders.csv Datei:

In [41]:
#Übersicht der Daten
df_orders.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09 00:00:00
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [ ]:
pruefe_tabelle(df_orders, "Orders", ["order_id"],
                kategorien=["order_status"],
                fremdschluessel={"customer_id": (df_customers, "customer_id")})
#Zeitspalten in DateTime umwandeln
df_orders = umwandlung_Zeitspalte(df_orders, ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"], "Orders")


Shape: (99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB

Fehlende Werte:
                                fehlend  prozent
order_approved_at                  160     0.16
order_delivered_carrier_date      1783     1.

## Bereinigung der Daten

## customer.csv Datei:

In [45]:
df_customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [44]:
pruefe_tabelle(
    df_customers, "customers", ["customer_id"],
    kategorien=["customer_state"]
)

Shape: (99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

Fehlende Werte:
 keine

Doppelte Zeilen: 0
Doppelte Schlüssel ['customer_id']: 0

Wertebereiche:
                             count          mean           std     min  \
customer_zip_code_prefix  99441.0  35137.474583  29797.938996  1003.0   

                              25%      50%      75%      max  
customer_zip_code_prefix  11347.0  24416.0  58900.0  99990.0  

Kategorien in customer_state:
 customer_state
SP    41746
RJ    12852
MG    11635
RS     5466


## Bereinigung der Daten:

## items.csv

In [47]:
df_items.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ce02b2c6e70d5b2fe24d103952d389bf,1,70d17ed46123e5b68dff23e113415d5a,3d871de0142ce09b7081e2b9d1733cb1,2017-08-03 13:45:11,54.90,12.72
1,4886c605fd1f7edd568a936814a015b7,1,bb09b0b5f3668effd704b1b2a83cae9b,6560211a19b47992c3666cc44a7e94c0,2017-12-22 02:31:10,189.00,8.58
2,2abac84637c5e0ddd2923f22ba61a96a,1,4ae634441e444ca4bc85903cafe98d73,92eb0f42c21942b6552362b9b114707d,2017-04-05 01:22:35,25.00,16.05
3,906dfb858a6afcaea1fb5cc0ed730147,1,33e77eecc23c25a65d10f627e5053617,7c67e1448b00f6e969d365cea6b010ab,2017-11-01 02:49:41,114.94,28.16
4,dcc495566353215aa81cab568f8fb2bf,1,6fd08d44046ab994b96ff38ad6fcfba1,d9a84e1403de8da0c3aa531d6d108ba6,2018-03-30 16:08:15,39.90,18.23
5,f8b6b27efdf0f73f9e9d978fc503c45f,1,2647221a6a477afe2d0268b378f1fdcd,d2374cbcbb3ca4ab1086534108cc3ab7,2018-07-10 17:31:38,39.90,19.47
6,d1c3d911ad830cab7d16ad8acfc36a46,2,05f0fe07929d35be0748fac56eecbeb2,c70c1b0d8ca86052f45a432a38b73958,2018-05-15 15:12:12,110.32,15.65
7,0ee0c320d2ceda401e28f8bc01e2fde2,1,5f24849289916bf82378a061a131fde9,be9a160a9011f627c3e121d076c1ec7b,2017-03-31 11:45:13,289.00,40.65
8,a7765f04ecc5f2e0683f08a97ee8174e,1,77d1f94d7eeaaeb36736fd1049e79eb3,8e6cc767478edae941d9bd9eb778d77a,2017-02-02 19:08:00,64.90,18.00
9,1c300b10263fefc0edc45c99921586a7,1,20a2613eb3b1c3819e733de3e91644de,1025f0e2d44d7041d6cf58b6550e0bfa,2018-04-02 21:15:25,35.00,8.29


In [48]:
pruefe_tabelle(
    df_items, "items", ["order_id", "order_item_id"],
    fremdschluessel={"order_id": (df_orders, "order_id"),
                     "product_id": (df_products, "product_id"),
                     "seller_id": (df_sellers, "seller_id")}
)
umwandlung_Zeitspalte(df_items, ["shipping_limit_date"], "Items")

Shape: (112875, 7)
<class 'pandas.DataFrame'>
RangeIndex: 112875 entries, 0 to 112874
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112875 non-null  str    
 1   order_item_id        112875 non-null  int64  
 2   product_id           112875 non-null  str    
 3   seller_id            112875 non-null  str    
 4   shipping_limit_date  112875 non-null  str    
 5   price                112875 non-null  float64
 6   freight_value        112875 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 18.4 MB

Fehlende Werte:
 keine

Doppelte Zeilen: 225
Doppelte Schlüssel ['order_id', 'order_item_id']: 225

Wertebereiche:
                   count        mean         std   min    25%    50%     75%  \
order_item_id  112875.0    1.197927    0.705163  1.00   1.00   1.00    1.00   
price          112875.0  120.663540  183.578651  0.85  39.90  74.99  134.90   
freight_va

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ce02b2c6e70d5b2fe24d103952d389bf,1,70d17ed46123e5b68dff23e113415d5a,3d871de0142ce09b7081e2b9d1733cb1,2017-08-03 13:45:11,54.90,12.72
1,4886c605fd1f7edd568a936814a015b7,1,bb09b0b5f3668effd704b1b2a83cae9b,6560211a19b47992c3666cc44a7e94c0,2017-12-22 02:31:10,189.00,8.58
2,2abac84637c5e0ddd2923f22ba61a96a,1,4ae634441e444ca4bc85903cafe98d73,92eb0f42c21942b6552362b9b114707d,2017-04-05 01:22:35,25.00,16.05
3,906dfb858a6afcaea1fb5cc0ed730147,1,33e77eecc23c25a65d10f627e5053617,7c67e1448b00f6e969d365cea6b010ab,2017-11-01 02:49:41,114.94,28.16
4,dcc495566353215aa81cab568f8fb2bf,1,6fd08d44046ab994b96ff38ad6fcfba1,d9a84e1403de8da0c3aa531d6d108ba6,2018-03-30 16:08:15,39.90,18.23
...,...,...,...,...,...,...,...
112870,1bdd4c66aa68cf72d2fc25b9f9e4bb48,1,33f17ed3c6906f7a7ae17a0843536db4,ea8482cd71df3c1969d7b9473ff13abc,2017-12-13 03:16:26,27.99,14.10
112871,5253650c645f3d453fc1dadecbe9d2c8,1,6d7a4492be45dfb8e38e762efe6614a4,ea8482cd71df3c1969d7b9473ff13abc,2017-12-05 12:18:29,27.99,15.10
112872,efd6e545a0e34fcc1b5a8fe60d9026ec,1,5a9db0654a2a5e1ee2b517023d04f279,709e16e2b25c7474d980076c6bfc4806,2017-11-30 14:19:23,83.90,12.09
112873,46cfd37753bbd00f768165cf4cb7ace1,1,ed38e11473775cde0d65f698aa0a4d27,3d871de0142ce09b7081e2b9d1733cb1,2017-11-01 01:49:09,89.00,10.05


## Bereinigung:

## Sellers

In [49]:
df_sellers.head(10)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
5,c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
6,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
7,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
8,768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
9,ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [50]:
pruefe_tabelle(
    df_sellers, "sellers", ["seller_id"],
    kategorien=["seller_state"]
)

Shape: (3095, 4)
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB

Fehlende Werte:
 keine

Doppelte Zeilen: 0
Doppelte Schlüssel ['seller_id']: 0

Wertebereiche:
                          count          mean          std     min     25%  \
seller_zip_code_prefix  3095.0  32291.059451  32713.45383  1001.0  7093.5   

                            50%      75%      max  
seller_zip_code_prefix  14940.0  64552.5  99730.0  

Kategorien in seller_state:
 seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB 